In [1]:
import numpy as np
from tanalysis import improcess, stitching
import tifffile as tiff
from tqdm import tqdm

dirname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Originals\24h_CXCL10_Conc10_z5_t8h.lif"



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.12.0 
torch version:  	2.8.0+cu126! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




In [2]:
imgs, names, info = improcess.imread(dirname, channel=0, tiles=True)
print(imgs[0].shape)

Reading submitted files: 100%|████████████████████████████████████████| 1/1 [00:07<00:00,  7.03s/it]

All files read!
(97, 6, 14, 1024, 1024)


In [3]:
positions = info['mosaic_position']
translations_list = stitching.translationComputation(imgs, positions, n=5)

Calculating translation vectors: 100%|██████████| 11/11 [05:29<00:00, 29.95s/it]

All vectors calculated!
[[910, -8, 890, 8]]


In [17]:
def image_reconstruction(imgs, positions, translations_list):
    '''
    This function reconstructs the mosaic image using translation vectors for the tiles. To determine the translation vectors, translationComputation function is used. The reconstructed image is formed by overlapping the stitched tiles
    and averaging the overlapping parts, resulting in a stitched image.

    Args:
        imgs: list of arrays of the images to stitch
        positions: position of each image in the resulting image
        translations_list: list of translations for the images
        ss, hk, h2, 

    Returns:
        res_img_list: list of resulting images
    '''
    grid_list, nrow, ncol = stitching.make_grid(imgs, positions)

    res_img_list = []
    for trans_set, grid in zip(translations_list, grid_list):
        abs_translations = {}
        minr=0
        minc=0
        drow, rr, dcol, rc = trans_set
        for row in np.arange(nrow):
            for col in np.arange(ncol):
                abs_translations[f'{row}{col}'] = [int(row*(drow+rr)+col*rr), int(row*rc+col*(dcol+rc))]
                minr_ = abs_translations[f'{row}{col}'][0]
                minc_ = abs_translations[f'{row}{col}'][1]
                if minr_<minr:
                    minr=minr_
                if minc_<minc:
                    minc=minc_
        H,W = imgs[0].shape[-2], imgs[0].shape[-1]
        Hmax,Wmax = abs_translations[f'{nrow-1}{ncol-1}']
        rerr = abs(minr)
        cerr = abs(minc)
        t_result = []
        for grid_t in tqdm(grid, 'Reconstructing timeframes'):
            z_result = []
            for z in np.arange(imgs[0].shape[-3]):
                tiles_list = []
                for trans in abs_translations:
                    result = np.zeros((Hmax+H+2*rerr, Wmax+W+2*cerr))
                    srow = abs_translations[trans][0]+rerr
                    scol = abs_translations[trans][1]+cerr
                    erow = srow+H
                    ecol = scol+W
                    result[srow:erow,scol:ecol] = grid_t[trans][z]+1
                    tiles_list.append(result)
                tiles_arr = np.asarray(tiles_list)
                div = (tiles_arr!=0).sum(axis=0)
                div[div==0]=1
                mean_result = tiles_arr.sum(axis=0)/div
                #del tiles_arr
                z_result.append(mean_result)
                del mean_result
            t_result.append(z_result)
            del z_result
        res_img = np.asarray(t_result)
        del t_result
        res_img_list.append(res_img)
    return res_img_list, div, tiles_list, tiles_arr

In [18]:
res_img_list, div, tiles_list, tiles_arr = image_reconstruction(imgs, positions, translations_list)

Reconstructing timeframes: 100%|██████████| 97/97 [06:09<00:00,  3.81s/it]


In [19]:
print(res_img_list[0].shape)
newname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Originals\test.tiff"
im = res_img_list[0]

tiff.imwrite(
    newname, 
    im[:,:,:,:].astype(np.uint16), 
    imagej=True,
    metadata={
        'axes':'TZYX'
    })

(97, 14, 1942, 2828)


c:\Users\pcanaleta\Documents\GitHub\tanalysis\.venv\Lib\site-packages\tifffile\tifffile.py:3817: UserWarning: <tifffile.TiffWriter 'test.tiff'> truncating ImageJ file
  warnings.warn(
